<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L7_Pre_training_Concepts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L7: Pre-training Concepts - How LLMs Learn Language

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 7 of 15

---

## 📚 Learning Objectives

By the end of this lesson, you will:
- Understand the three main pre-training objectives for LLMs
- Implement Masked Language Modeling (MLM) used in BERT
- Implement Causal Language Modeling (CLM) used in GPT
- Understand Next Sentence Prediction (NSP) for sentence relationships
- Compare different pre-training strategies and their use cases
- Build custom pre-training tasks for specific applications

---

## 🧠 Concept: Pre-training - The Foundation of LLMs

**Pre-training** is the process of teaching a language model to understand language by training on massive amounts of text data.

### Why Pre-training?

**The Challenge:**
- Labeled data is expensive and limited
- Language understanding requires broad knowledge
- Task-specific training from scratch is inefficient

**The Solution:**
- Train on unlabeled text (billions of words)
- Learn general language patterns
- Fine-tune for specific tasks later

### Three Main Pre-training Objectives

1. **Masked Language Modeling (MLM)**
   - Hide random words, predict them from context
   - Used by: BERT, RoBERTa, ALBERT
   - Bidirectional understanding

2. **Causal Language Modeling (CLM)**
   - Predict next word given previous words
   - Used by: GPT, GPT-2, GPT-3, LLaMA
   - Left-to-right generation

3. **Next Sentence Prediction (NSP)**
   - Predict if two sentences follow each other
   - Used by: BERT (original)
   - Sentence relationship understanding

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install torch transformers matplotlib numpy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM
import numpy as np
import matplotlib.pyplot as plt
import random

print("✅ Libraries imported successfully!")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"🔧 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 🎭 Part 1: Masked Language Modeling (MLM)

**MLM** is like a fill-in-the-blank exercise. The model learns to predict masked words using context from both left and right.

### How It Works:

1. **Original:** "The cat sat on the mat"
2. **Masked:** "The [MASK] sat on the mat"
3. **Predict:** "cat"

### Masking Strategy (BERT):
- 15% of tokens are selected for masking
- Of those 15%:
  - 80% replaced with [MASK]
  - 10% replaced with random token
  - 10% kept unchanged

### Why This Strategy?
- Forces model to use context
- Prevents overfitting to [MASK] token
- Learns robust representations

---

In [ ]:
# Step 2: Demonstrate MLM with Pre-trained BERT

# Load BERT model and tokenizer
print("📥 Loading BERT model for MLM...\n")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

# Sample sentences
sentences = [
    "The cat sat on the [MASK].",
    "Paris is the [MASK] of France.",
    "Machine learning is a subset of [MASK] intelligence."
]

print("🎭 Masked Language Modeling Examples:\n")

for sent in sentences:
    # Tokenize
    inputs = tokenizer(sent, return_tensors="pt")
    
    # Find mask position
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = outputs.logits
    
    # Get top 5 predictions
    mask_token_logits = predictions[0, mask_token_index, :]
    top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
    
    print(f"📝 Sentence: {sent}")
    print(f"🎯 Top 5 predictions:")
    for i, token_id in enumerate(top_5_tokens, 1):
        token = tokenizer.decode([token_id])
        print(f"   {i}. {token}")
    print()

print("💡 Notice how the model uses bidirectional context to make predictions!")

## 🔮 Part 2: Causal Language Modeling (CLM)

**CLM** predicts the next word given all previous words. This is how GPT models work!

### How It Works:

1. **Input:** "The cat sat on"
2. **Predict:** "the"
3. **Next Input:** "The cat sat on the"
4. **Predict:** "mat"

### Key Characteristics:
- **Unidirectional:** Only looks at previous words (left-to-right)
- **Autoregressive:** Uses its own predictions as input
- **Generation:** Perfect for text generation tasks

### Causal Masking:
```
Position:  0  1  2  3
Token:    The cat sat on
Mask:     ✓  ✓  ✓  ✓   (position 0 sees only "The")
          ✗  ✓  ✓  ✓   (position 1 sees "The cat")
          ✗  ✗  ✓  ✓   (position 2 sees "The cat sat")
          ✗  ✗  ✗  ✓   (position 3 sees "The cat sat on")
```

---

In [ ]:
# Step 3: Demonstrate CLM with GPT-2

# Load GPT-2 model
print("📥 Loading GPT-2 model for CLM...\n")
tokenizer_gpt = AutoTokenizer.from_pretrained("gpt2")
model_gpt = AutoModelForCausalLM.from_pretrained("gpt2")

# Set pad token
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token

# Sample prompts
prompts = [
    "The future of artificial intelligence",
    "Once upon a time in a distant galaxy",
    "The key to successful machine learning is"
]

print("🔮 Causal Language Modeling Examples:\n")

for prompt in prompts:
    # Tokenize
    inputs = tokenizer_gpt(prompt, return_tensors="pt")
    
    # Generate
    with torch.no_grad():
        outputs = model_gpt.generate(
            inputs["input_ids"],
            max_length=30,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer_gpt.eos_token_id
        )
    
    generated_text = tokenizer_gpt.decode(outputs[0], skip_special_tokens=True)
    
    print(f"📝 Prompt: {prompt}")
    print(f"🎯 Generated: {generated_text}")
    print()

print("💡 CLM generates text by predicting one token at a time, left-to-right!")

## 🔗 Part 3: Next Sentence Prediction (NSP)

**NSP** teaches models to understand relationships between sentences. Is sentence B a natural continuation of sentence A?

### How It Works:

**Example 1 (IsNext):**
- Sentence A: "The cat sat on the mat."
- Sentence B: "It was very comfortable."
- Label: **IsNext** ✓

**Example 2 (NotNext):**
- Sentence A: "The cat sat on the mat."
- Sentence B: "The stock market crashed yesterday."
- Label: **NotNext** ✗

### Why NSP?
- Understand document structure
- Learn sentence relationships
- Improve performance on QA and NLI tasks

### Training Data Creation:
- 50% of the time: Use actual next sentence (IsNext)
- 50% of the time: Use random sentence (NotNext)

---

In [ ]:
# Step 4: Demonstrate NSP Concept

def create_nsp_examples():
    """
    Create NSP training examples.
    """
    # Sample document
    document = [
        "The transformer architecture was introduced in 2017.",
        "It revolutionized natural language processing.",
        "Transformers use self-attention mechanisms.",
        "This allows them to process sequences in parallel.",
        "The weather today is sunny and warm.",
        "Many people enjoy outdoor activities in such weather."
    ]
    
    examples = []
    
    # Create IsNext examples
    for i in range(len(document) - 1):
        if i < 3:  # First 4 sentences are related
            examples.append({
                "sentence_a": document[i],
                "sentence_b": document[i + 1],
                "label": "IsNext"
            })
    
    # Create NotNext examples
    examples.append({
        "sentence_a": document[0],
        "sentence_b": document[4],  # Unrelated sentence
        "label": "NotNext"
    })
    
    examples.append({
        "sentence_a": document[2],
        "sentence_b": document[5],  # Unrelated sentence
        "label": "NotNext"
    })
    
    return examples

# Generate examples
nsp_examples = create_nsp_examples()

print("🔗 Next Sentence Prediction Examples:\n")

for i, example in enumerate(nsp_examples, 1):
    print(f"Example {i}:")
    print(f"  Sentence A: {example['sentence_a']}")
    print(f"  Sentence B: {example['sentence_b']}")
    print(f"  Label: {example['label']} {'✓' if example['label'] == 'IsNext' else '✗'}")
    print()

print("💡 NSP helps models understand document structure and coherence!")

## 📊 Part 4: Comparing Pre-training Objectives

Let's compare the three pre-training methods:

| Aspect | MLM (BERT) | CLM (GPT) | NSP (BERT) |
|--------|------------|-----------|------------|
| **Direction** | Bidirectional | Unidirectional | Bidirectional |
| **Task** | Fill in blanks | Predict next word | Sentence relationship |
| **Best For** | Understanding | Generation | Document tasks |
| **Context** | Full sentence | Previous words only | Sentence pairs |
| **Examples** | BERT, RoBERTa | GPT, LLaMA | BERT (original) |

### When to Use Each:

**MLM (Masked Language Modeling):**
- Classification tasks
- Named Entity Recognition
- Question Answering
- When you need bidirectional context

**CLM (Causal Language Modeling):**
- Text generation
- Chatbots
- Code generation
- Creative writing

**NSP (Next Sentence Prediction):**
- Document classification
- Sentence ordering
- Coherence detection
- Note: Less common in modern models (RoBERTa removed it)

---

In [ ]:
# Step 5: Visualize Pre-training Objectives

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MLM Visualization
ax1 = axes[0]
ax1.set_title('Masked Language Modeling (MLM)', fontsize=14, fontweight='bold')
tokens = ['The', '[MASK]', 'sat', 'on', 'the', 'mat']
colors = ['lightblue' if t != '[MASK]' else 'orange' for t in tokens]
y_pos = np.arange(len(tokens))
ax1.barh(y_pos, [1]*len(tokens), color=colors, edgecolor='black', linewidth=2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(tokens)
ax1.set_xlabel('Attention (Bidirectional)')
ax1.set_xlim(0, 1.2)
ax1.text(0.6, len(tokens)-0.5, '← Bidirectional →', fontsize=10, ha='center')
ax1.grid(axis='x', alpha=0.3)

# CLM Visualization
ax2 = axes[1]
ax2.set_title('Causal Language Modeling (CLM)', fontsize=14, fontweight='bold')
tokens = ['The', 'cat', 'sat', 'on', 'the', '?']
colors = ['lightgreen'] * (len(tokens)-1) + ['orange']
y_pos = np.arange(len(tokens))
widths = [0.2, 0.4, 0.6, 0.8, 1.0, 1.0]
ax2.barh(y_pos, widths, color=colors, edgecolor='black', linewidth=2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(tokens)
ax2.set_xlabel('Attention (Left-to-Right)')
ax2.set_xlim(0, 1.2)
ax2.text(0.6, len(tokens)-0.5, '← Causal →', fontsize=10, ha='center')
ax2.grid(axis='x', alpha=0.3)

# NSP Visualization
ax3 = axes[2]
ax3.set_title('Next Sentence Prediction (NSP)', fontsize=14, fontweight='bold')
ax3.text(0.5, 0.7, 'Sentence A:\n"The cat sat on the mat."', 
         ha='center', va='center', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black', linewidth=2))
ax3.text(0.5, 0.3, 'Sentence B:\n"It was comfortable."', 
         ha='center', va='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightgreen', edgecolor='black', linewidth=2))
ax3.arrow(0.5, 0.55, 0, -0.15, head_width=0.05, head_length=0.03, fc='orange', ec='orange', linewidth=3)
ax3.text(0.5, 0.1, 'IsNext? ✓ or ✗', ha='center', fontsize=12, fontweight='bold', color='orange')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')

plt.tight_layout()
plt.show()

print("📊 Visualization complete!")
print("\n💡 Key Differences:")
print("   - MLM: Bidirectional context, predicts masked tokens")
print("   - CLM: Unidirectional context, predicts next token")
print("   - NSP: Sentence-level task, predicts relationships")

## 🔬 Exercises

### Exercise 1: Custom MLM
Create your own masked sentences and test BERT's predictions:
```python
custom_sentences = [
    "Python is a popular [MASK] language.",
    "The [MASK] is the largest planet in our solar system.",
    "Deep learning requires large amounts of [MASK]."
]
# Test with the MLM code above
```

### Exercise 2: CLM Temperature
Experiment with different temperature values in CLM generation:
- temperature=0.1 (more deterministic)
- temperature=1.0 (balanced)
- temperature=2.0 (more random)

How does it affect the generated text?

### Exercise 3: NSP Dataset
Create a small NSP dataset from a Wikipedia article:
1. Split article into sentences
2. Create IsNext pairs from consecutive sentences
3. Create NotNext pairs from random sentences
4. Calculate the ratio of coherent vs incoherent pairs

### Exercise 4: Compare Models
Load both BERT and GPT-2, compare:
- Model sizes (number of parameters)
- Inference speed
- Memory usage
- Best use cases

### Exercise 5: Custom Pre-training Task
Design a new pre-training objective for a specific domain (e.g., code, medical text):
- What would you mask or predict?
- Why would this help the model?
- What downstream tasks would benefit?

---

## 🎓 Key Takeaways

1. **Pre-training** teaches models general language understanding from unlabeled data
2. **MLM (Masked Language Modeling)** uses bidirectional context to predict masked words
3. **CLM (Causal Language Modeling)** predicts next words autoregressively
4. **NSP (Next Sentence Prediction)** learns sentence relationships
5. **BERT** uses MLM + NSP for understanding tasks
6. **GPT** uses CLM for generation tasks
7. **Modern trends**: CLM is more popular (GPT-3, LLaMA), NSP often removed (RoBERTa)
8. **Pre-training** enables transfer learning and few-shot learning

### Pre-training vs Fine-tuning

```
Pre-training (Unsupervised):
  Massive unlabeled data → General language model

Fine-tuning (Supervised):
  Small labeled data → Task-specific model
```

---

## 📚 Additional Resources

### Papers
- **"BERT: Pre-training of Deep Bidirectional Transformers"** (Devlin et al., 2018)
- **"Language Models are Unsupervised Multitask Learners"** (Radford et al., 2019) - GPT-2
- **"RoBERTa: A Robustly Optimized BERT Pretraining Approach"** (Liu et al., 2019)

### Tutorials
- [HuggingFace Pre-training Guide](https://huggingface.co/docs/transformers/training)
- [The Illustrated BERT](http://jalammar.github.io/illustrated-bert/)
- [GPT-2 Explained](http://jalammar.github.io/illustrated-gpt2/)

### Interactive
- [BERT Visualizer](https://github.com/jessevig/bertviz)
- [GPT-2 Playground](https://transformer.huggingface.co/)

---

## ➡️ Next Lesson

**L8: Tokenizers Deep Dive** - Learn about BPE, WordPiece, and SentencePiece tokenization algorithms

---

**Congratulations! You now understand how LLMs learn language! 🎉**

*You've learned the foundation of modern NLP: pre-training objectives that power BERT, GPT, and beyond.*